# Job Market Dataset Profiling

Lightweight exploration for the raw Kaggle dataset `uom190346a/ai-powered-job-market-insights`.

This notebook is intentionally exploratory. Pipeline logic belongs under `src/`.

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd()
if (cwd / "src" / "ingestion").exists():
    PROJECT_ROOT = cwd
elif cwd.name == "notebooks" and (cwd.parent / "src" / "ingestion").exists():
    PROJECT_ROOT = cwd.parent
elif (cwd / "projects" / "02-job-market-analytics" / "src" / "ingestion").exists():
    PROJECT_ROOT = cwd / "projects" / "02-job-market-analytics"
else:
    raise RuntimeError("Launch this notebook from the project directory, notebook directory, or repository root.")

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ingestion.raw_inventory import discover_raw_files, get_raw_data_dir, list_supported_data_files
from processing.bronze.profiling import select_main_csv_file

raw_dir = get_raw_data_dir()
raw_dir

WindowsPath('C:/Users/willi/OneDrive/Área de Trabalho/PhaifferTech/repos/data-engineering-portfolio/projects/02-job-market-analytics/data/bronze/raw/ai_powered_job_market_insights')

## Raw File Discovery

In [2]:
inventory = discover_raw_files(raw_dir)
inventory

{'raw_directory': 'data/bronze/raw/ai_powered_job_market_insights',
 'supported_extensions': ['.csv'],
 'file_count': 2,
 'supported_data_file_count': 1,
 'files': [{'path': 'data/bronze/raw/ai_powered_job_market_insights/.complete/datasets/uom190346a/ai-powered-job-market-insights/1/bundle.complete',
   'name': 'bundle.complete',
   'extension': '.complete',
   'size_bytes': 0,
   'is_supported_data_file': False,
   'is_selected_for_bronze_profiling': False},
  {'path': 'data/bronze/raw/ai_powered_job_market_insights/ai_job_market_insights.csv',
   'name': 'ai_job_market_insights.csv',
   'extension': '.csv',
   'size_bytes': 47412,
   'is_supported_data_file': True,
   'is_selected_for_bronze_profiling': True}],
 'supported_data_files': ['data/bronze/raw/ai_powered_job_market_insights/ai_job_market_insights.csv']}

In [3]:
csv_files = list_supported_data_files(raw_dir)
csv_files

[WindowsPath('C:/Users/willi/OneDrive/Área de Trabalho/PhaifferTech/repos/data-engineering-portfolio/projects/02-job-market-analytics/data/bronze/raw/ai_powered_job_market_insights/ai_job_market_insights.csv')]

## Dataset Loading

In [4]:
import pandas as pd

main_csv = select_main_csv_file(csv_files)
df = pd.read_csv(main_csv)
main_csv

WindowsPath('C:/Users/willi/OneDrive/Área de Trabalho/PhaifferTech/repos/data-engineering-portfolio/projects/02-job-market-analytics/data/bronze/raw/ai_powered_job_market_insights/ai_job_market_insights.csv')

In [5]:
df.head()

,Job_Title,Industry,Company_Size,Location,AI_Adoption_Level,Automation_Risk,Required_Skills,Salary_USD,Remote_Friendly,Job_Growth_Projection
0,Cybersecurity Analyst,Entertainment,Small,Dubai,Medium,High,UX/UI Design,111392.165243,Yes,Growth
1,Marketing Specialist,Technology,Large,Singapore,Medium,High,Marketing,93792.562466,No,Decline
2,AI Researcher,Technology,Large,Singapore,Medium,High,UX/UI Design,107170.263069,Yes,Growth
3,Sales Manager,Retail,Small,Berlin,Low,High,Project Management,93027.953758,No,Growth
4,Cybersecurity Analyst,Entertainment,Small,Tokyo,Low,Low,JavaScript,87752.922171,Yes,Decline


In [6]:
df.shape

(500, 10)

In [7]:
list(df.columns)

['Job_Title',
 'Industry',
 'Company_Size',
 'Location',
 'AI_Adoption_Level',
 'Automation_Risk',
 'Required_Skills',
 'Salary_USD',
 'Remote_Friendly',
 'Job_Growth_Projection']

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Job_Title              500 non-null    str    
 1   Industry               500 non-null    str    
 2   Company_Size           500 non-null    str    
 3   Location               500 non-null    str    
 4   AI_Adoption_Level      500 non-null    str    
 5   Automation_Risk        500 non-null    str    
 6   Required_Skills        500 non-null    str    
 7   Salary_USD             500 non-null    float64
 8   Remote_Friendly        500 non-null    str    
 9   Job_Growth_Projection  500 non-null    str    
dtypes: float64(1), str(9)
memory usage: 72.0 KB


## Basic Quality Checks

In [9]:
df.isna().sum().sort_values(ascending=False)

Job_Title                0
Industry                 0
Company_Size             0
Location                 0
AI_Adoption_Level        0
Automation_Risk          0
Required_Skills          0
Salary_USD               0
Remote_Friendly          0
Job_Growth_Projection    0
dtype: int64

In [10]:
df.duplicated().sum()

np.int64(0)

## Initial Observations

- Review the raw columns before defining Silver naming and type rules.
- Use null counts and duplicate counts to decide which checks belong in future quality gates.
- Keep business semantics out of Bronze; dimensional modeling should start after the raw schema is confirmed.